# AutoMode: four-GPU SSH launcher

**Workflow:** Claude patches the repo → you push and copy the exact commit SHA → this notebook deploys that snapshot to four rented GPU workers → preflight → pilot → 15 core runs → download results.

**Required first:** give Claude `CLAUDE_SETUP_PROMPT.md` and `RUNNER_CONTRACT.md` from the kit, with your repository open. The uploaded old package does **not** implement the required `automode.rebuttal` CLI. This notebook deliberately refuses to launch that old code.

Use this notebook in **Colab connected to a local runtime on your laptop**, or ordinary Jupyter. No local GPU is needed. Compute runs entirely on the rented machines. Jobs are detached and persist after notebook/SSH disconnects, as long as the rental machines remain running. This notebook does not rent or terminate machines.

Official setup: https://research.google.com/colaboratory/local-runtimes.html . On your laptop, install Jupyter and run the command in the README, then in Colab select **Connect → Connect to local runtime**. Free managed Colab runtimes restrict remote-control workloads; this kit is designed for your own runtime. Do not use `drive.mount()` with this setup. Download results to a local disk folder, then back them up before releasing rentals.

Each worker needs Linux, SSH/SCP, bash, git-independent Python 3.10+ with `venv`, and an existing GPU-compatible PyTorch/CUDA environment. Prefer identical 96GB GPUs and ~200GB free disk per worker (especially worker0, which saves four full checkpoints). No driver or CUDA installation is attempted.


## 1. Set your repository, commit and SSH endpoints

Enter a **40-character commit SHA** from the updated GitHub repo, not `main`. For a private repo, use your laptop's normal Git authentication (SSH agent or credential manager). Do not put a token in REPO_URL.

For four separate machines use GPU `0` on each. For one machine with four GPUs, repeat its host/port and use GPU `0`, `1`, `2`, `3`. The notebook rejects repeated endpoint/device pairs and later compares physical GPU UUIDs.

Before running, connect once to each endpoint from your terminal, check its host fingerprint against the rental provider, and add it to your known_hosts using normal SSH. This notebook never disables host-key verification. SSH agent authentication works; alternatively set an absolute `key_path` below. Encrypted keys should be loaded into your SSH agent. Password-only SSH and jump-host routing are not configured here.


In [ ]:
from pathlib import Path
REPO_URL = "git@github.com:YOUR_USERNAME/YOUR_REPOSITORY.git"
COMMIT = "PASTE_THE_FULL_40_CHARACTER_COMMIT_SHA"
CAMPAIGN = "automode_aaai_rebuttal_v1"
LOCAL_ROOT = Path.home() / "automode_rebuttal_control"
CORE_BUDGET_HOURS = 14.0
LAUNCH_CORE = False  # Set True only after reviewing all four pilot reports.

WORKERS = [
    dict(id=0, host="GPU_HOST_0", port=22, user="root", gpu="0", key_path="", remote_base="/workspace/automode_rebuttal", python="python3"),
    dict(id=1, host="GPU_HOST_1", port=22, user="root", gpu="0", key_path="", remote_base="/workspace/automode_rebuttal", python="python3"),
    dict(id=2, host="GPU_HOST_2", port=22, user="root", gpu="0", key_path="", remote_base="/workspace/automode_rebuttal", python="python3"),
    dict(id=3, host="GPU_HOST_3", port=22, user="root", gpu="0", key_path="", remote_base="/workspace/automode_rebuttal", python="python3"),
]


## 2. Load controller helpers and the core manifest

Standard-library Python plus your existing local `ssh`, `scp`, and `git` executables. Credentials stay out of notebook source and outputs. The embedded supervisor matches `tools/remote_job.py` in the kit.


In [ ]:
CORE_MANIFEST = {'schema_version': 1, 'protocol_version': 'automode_rebuttal_corrected_v1', 'expected_eval_count': 1319, 'minimum_vram_gib': 70, 'common': {'model_name': 'google/gemma-2-9b', 'model_revision': None, 'dataset_revision': None, 'train_track': 'gsm8k', 'eval_benchmarks': ['gsm8k'], 'dtype': 'bfloat16', 'gradient_checkpointing': True, 'epochs': 2, 'learning_rate': 2e-05, 'lr_scheduler': 'linear', 'warmup_ratio': 0.03, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'train_batch_size': 4, 'grad_accum_steps': 8, 'max_train_samples': None, 'max_eval_samples': None, 'max_source_len': 384, 'max_target_len': 128, 'max_new_tokens': 256, 'eval_batch_size': 8, 'do_sample': True, 'temperature': 0.7, 'top_p': 0.95, 'sampling_k': 3, 'evaluation_seed': 20261029, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'lora_target_modules': ['q_proj', 'v_proj'], 'dynamic_updates': 10, 'dynamic_threshold': 10, 'importance_signal': 'grad_norm', 'score_collection': 'optimizer_step', 'switching_mode': 'percentile', 'save_model': False, 'save_eval_predictions': True, 'use_8bit_optim_for_full_ft': False}, 'jobs': [{'job_id': 's8_automode', 'worker_id': 0, 'overrides': {'method': 'automode', 'seed': 8, 'data_seed': 8, 'save_model': True}}, {'job_id': 's8_select_once', 'worker_id': 0, 'overrides': {'method': 'select_once', 'seed': 8, 'data_seed': 8, 'save_model': True}}, {'job_id': 's8_fixed_mixed', 'worker_id': 0, 'overrides': {'method': 'fixed_mixed', 'seed': 8, 'data_seed': 8, 'save_model': True, 'fixed_base_layers': 37, 'allocation_seed': 8}}, {'job_id': 's8_static_qv', 'worker_id': 0, 'overrides': {'method': 'static_qv', 'seed': 8, 'data_seed': 8, 'save_model': True}}, {'job_id': 's25_automode', 'worker_id': 1, 'overrides': {'method': 'automode', 'seed': 25, 'data_seed': 25, 'save_model': False}}, {'job_id': 's25_select_once', 'worker_id': 1, 'overrides': {'method': 'select_once', 'seed': 25, 'data_seed': 25, 'save_model': False}}, {'job_id': 's25_fixed_mixed', 'worker_id': 1, 'overrides': {'method': 'fixed_mixed', 'seed': 25, 'data_seed': 25, 'save_model': False, 'fixed_base_layers': 37, 'allocation_seed': 25}}, {'job_id': 's25_static_qv', 'worker_id': 1, 'overrides': {'method': 'static_qv', 'seed': 25, 'data_seed': 25, 'save_model': False}}, {'job_id': 's42_automode', 'worker_id': 2, 'overrides': {'method': 'automode', 'seed': 42, 'data_seed': 42, 'save_model': False}}, {'job_id': 's42_select_once', 'worker_id': 2, 'overrides': {'method': 'select_once', 'seed': 42, 'data_seed': 42, 'save_model': False}}, {'job_id': 's42_fixed_mixed', 'worker_id': 2, 'overrides': {'method': 'fixed_mixed', 'seed': 42, 'data_seed': 42, 'save_model': False, 'fixed_base_layers': 37, 'allocation_seed': 42}}, {'job_id': 's42_static_qv', 'worker_id': 2, 'overrides': {'method': 'static_qv', 'seed': 42, 'data_seed': 42, 'save_model': False}}, {'job_id': 's8_lora_r32', 'worker_id': 3, 'overrides': {'method': 'lora', 'seed': 8, 'data_seed': 8, 'lora_r': 32, 'lora_alpha': 64, 'save_model': True}}, {'job_id': 's25_lora_r32', 'worker_id': 3, 'overrides': {'method': 'lora', 'seed': 25, 'data_seed': 25, 'lora_r': 32, 'lora_alpha': 64, 'save_model': False}}, {'job_id': 's42_lora_r32', 'worker_id': 3, 'overrides': {'method': 'lora', 'seed': 42, 'data_seed': 42, 'lora_r': 32, 'lora_alpha': 64, 'save_model': False}}]}
REMOTE_JOB_SOURCE = '#!/usr/bin/env python3\n"""Detached, single-job supervisor for Linux SSH workers. Standard library only."""\nimport argparse\nimport datetime as dt\nimport fcntl\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport time\nimport uuid\n\n\ndef now():\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    tmp = path.with_name(path.name + f\'.{os.getpid()}.tmp\')\n    tmp.write_text(json.dumps(obj, indent=2))\n    os.replace(tmp, path)\n\n\ndef read_json(path):\n    try:\n        return json.loads(Path(path).read_text())\n    except FileNotFoundError:\n        return {}\n\n\ndef lock_held(root):\n    """File lock is authoritative; no external PID-based signaling/reuse risk."""\n    with (Path(root) / \'running.lock\').open(\'a\') as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n            return False\n        except BlockingIOError:\n            return True\n\n\ndef terminate_child(child):\n    if child.poll() is not None:\n        return\n    try:\n        os.killpg(child.pid, signal.SIGTERM)\n        child.wait(timeout=20)\n    except subprocess.TimeoutExpired:\n        os.killpg(child.pid, signal.SIGKILL)\n        child.wait()\n    except ProcessLookupError:\n        pass\n\n\ndef supervise(job_dir):\n    job_dir = Path(job_dir)\n    with (job_dir / \'running.lock\').open(\'a\') as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            return 79\n        request = read_json(job_dir / \'request.json\')\n        state = dict(status=\'running\', pid=os.getpid(), run_token=uuid.uuid4().hex,\n                     started_at=now(), request_digest=request[\'request_digest\'])\n        write_json(job_dir / \'status.json\', state)\n        stopped = [False]\n        def stop_handler(*_):\n            stopped[0] = True\n        signal.signal(signal.SIGTERM, stop_handler)\n        signal.signal(signal.SIGINT, stop_handler)\n        child = None\n        try:\n            env = os.environ.copy()\n            additions = json.loads(Path(request[\'env_file\']).read_text())\n            if not all(isinstance(k, str) and isinstance(v, str) for k, v in additions.items()):\n                raise ValueError(\'Environment values must be strings\')\n            env.update(additions)\n            deadline = time.monotonic() + request[\'timeout_hours\'] * 3600\n            with (job_dir / \'output.log\').open(\'a\', buffering=1) as log:\n                child = subprocess.Popen(request[\'command\'], cwd=request[\'cwd\'], env=env,\n                                         stdin=subprocess.DEVNULL, stdout=log,\n                                         stderr=subprocess.STDOUT, start_new_session=True)\n                state[\'child_pid\'] = child.pid\n                while child.poll() is None:\n                    state[\'heartbeat_at\'] = now()\n                    write_json(job_dir / \'status.json\', state)\n                    stop_request = read_json(job_dir / \'stop_request.json\')\n                    if stop_request.get(\'run_token\') == state[\'run_token\']:\n                        stopped[0] = True\n                    if stopped[0] or time.monotonic() >= deadline:\n                        state[\'status\'] = \'stopped\' if stopped[0] else \'timed_out\'\n                        terminate_child(child)\n                        break\n                    time.sleep(1)\n                state[\'returncode\'] = child.wait()\n            if state[\'status\'] == \'running\':\n                state[\'status\'] = \'completed\' if state[\'returncode\'] == 0 else \'failed\'\n        except Exception as exc:\n            if child is not None:\n                terminate_child(child)\n            state.update(status=\'failed\', error=f\'{type(exc).__name__}: {exc}\')\n        state[\'finished_at\'] = now()\n        write_json(job_dir / \'status.json\', state)\n        return 0 if state[\'status\'] == \'completed\' else 1\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=[\'start\', \'status\', \'stop\', \'_run\'])\n    parser.add_argument(\'--job-dir\', required=True)\n    parser.add_argument(\'--cwd\')\n    parser.add_argument(\'--env-file\')\n    parser.add_argument(\'--timeout-hours\', type=float, default=16)\n    parser.add_argument(\'--retry\', action=\'store_true\')\n    argv = sys.argv[1:]\n    split = argv.index(\'--\') if \'--\' in argv else len(argv)\n    command = argv[split + 1:] if split < len(argv) else []\n    args = parser.parse_args(argv[:split])\n    root = Path(args.job_dir).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    if args.action == \'_run\':\n        return supervise(root)\n    state = read_json(root / \'status.json\')\n    if args.action == \'status\':\n        if state.get(\'status\') == \'running\' and not lock_held(root):\n            state = read_json(root / \'status.json\')\n            if state.get(\'status\') == \'running\':\n                state[\'status\'] = \'interrupted\'\n        print(json.dumps(state or {\'status\': \'not_started\'}))\n        return 0\n    if args.action == \'stop\':\n        if state.get(\'status\') == \'running\' and lock_held(root):\n            write_json(root / \'stop_request.json\', {\'run_token\': state[\'run_token\'], \'requested_at\': now()})\n            print(json.dumps({\'status\': \'stop_requested\'}))\n        else:\n            print(json.dumps({\'status\': \'not_running\'}))\n        return 0\n    if not command or not args.cwd or not args.env_file or args.timeout_hours <= 0:\n        parser.error(\'start requires --cwd, --env-file, positive timeout and -- COMMAND\')\n    request = dict(command=command, cwd=str(Path(args.cwd).resolve()),\n                   env_file=str(Path(args.env_file).resolve()), timeout_hours=args.timeout_hours)\n    request[\'request_digest\'] = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()\n    with (root / \'launch.lock\').open(\'a\') as launch_lock:\n        fcntl.flock(launch_lock, fcntl.LOCK_EX)\n        old_request = read_json(root / \'request.json\')\n        state = read_json(root / \'status.json\')\n        if old_request and old_request.get(\'request_digest\') != request[\'request_digest\']:\n            raise RuntimeError(\'Job specification changed: use a new job directory\')\n        if state.get(\'status\') == \'running\' and lock_held(root):\n            print(json.dumps(state)); return 0\n        if state.get(\'status\') == \'completed\':\n            print(json.dumps(state)); return 0\n        if state and not args.retry:\n            raise RuntimeError(\'Previous attempt did not complete. Inspect logs; use --retry explicitly.\')\n        write_json(root / \'request.json\', request)\n        if state:\n            write_json(root / f"previous_status_{time.time_ns()}.json", state)\n            (root / \'status.json\').unlink(missing_ok=True)\n        with (root / \'supervisor.log\').open(\'a\') as log:\n            proc = subprocess.Popen([sys.executable, str(Path(__file__).resolve()), \'_run\',\n                                     \'--job-dir\', str(root)], stdin=subprocess.DEVNULL,\n                                    stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        for _ in range(30):\n            state = read_json(root / \'status.json\')\n            if state:\n                print(json.dumps(state)); return 0\n            if proc.poll() is not None:\n                raise RuntimeError(\'Supervisor failed to start; inspect supervisor.log\')\n            time.sleep(.1)\n        raise RuntimeError(\'No startup acknowledgement. Check status before retrying.\')\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'


In [ ]:
import os, re, json, shlex, subprocess, hashlib, tarfile, io, shutil, time, statistics, csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, FileLink

q = shlex.quote
REQUIRED_METHODS = {"automode", "lora", "select_once", "fixed_mixed", "static_qv"}

def checked(argv, *, input_bytes=None, timeout=120):
    p = subprocess.run(argv, input=input_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
    if p.returncode:
        raise RuntimeError(f"Command failed ({p.returncode}): {p.stderr.decode(errors='replace')[-6000:]}")
    return p.stdout.decode(errors="replace")

def ssh_prefix(w):
    a = ["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=yes", "-o", "ConnectTimeout=15", "-p", str(w["port"])]
    if w["key_path"]: a += ["-i", str(Path(w["key_path"]).expanduser())]
    return a + [f'{w["user"]}@{w["host"]}']

def remote(w, argv, *, data=None, timeout=120):
    return checked(ssh_prefix(w) + [shlex.join([str(x) for x in argv])], input_bytes=data, timeout=timeout)

def remote_py(w, script, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode()
    return remote(w, [w["python"], "-c", script], data=data, timeout=timeout)

def transfer(w, source, dest, download=False):
    a = ["scp", "-q", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=yes", "-o", "ConnectTimeout=15", "-P", str(w["port"])]
    if w["key_path"]: a += ["-i", str(Path(w["key_path"]).expanduser())]
    endpoint=f'{w["user"]}@{w["host"]}:'
    if download: a += [endpoint+str(source), str(dest)]
    else: a += [str(source), endpoint+str(dest)]
    checked(a, timeout=600)

def each_worker(fn):
    results = {}
    errors = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(fn,w):w["id"] for w in WORKERS}
        for f in as_completed(futures):
            wid=futures[f]
            try: results[wid]=f.result()
            except Exception as e: errors.append((wid,str(e)))
    for wid,err in errors: print(f"Worker {wid}: {err}")
    if errors: raise RuntimeError("Some workers failed; inspect the messages above before continuing.")
    return results

def read_remote_json(w, path):
    return json.loads(remote_py(w, "import json,sys,pathlib; p=pathlib.Path(json.load(sys.stdin)['path']); print(p.read_text())", {"path":str(path)}))

def job_status(w, phase):
    return json.loads(remote(w,[w["python"],w["root"]+"/remote_job.py","status","--job-dir",w["root"]+"/jobs/"+phase]))

def require_completed(phase):
    states=each_worker(lambda w:job_status(w,phase))
    display([{ "worker":wid, **s } for wid,s in sorted(states.items())])
    if not all(s.get("status")=="completed" for s in states.values()):
        raise RuntimeError(f"{phase} has not completed successfully on every worker. Use the monitoring cell, then rerun this cell.")
    return states

def launch(w, phase, argv, timeout_hours, retry=False):
    cmd=[w["python"],w["root"]+"/remote_job.py","start","--job-dir",w["root"]+"/jobs/"+phase,
         "--cwd",w["root"]+"/repo","--env-file",w["root"]+"/worker_env.json","--timeout-hours",str(timeout_hours)]
    if retry:cmd.append("--retry")
    return json.loads(remote(w,cmd+["--"]+argv))

def cli(w, command, extra=None):
    return [w["root"]+"/venv/bin/python","-m","automode.rebuttal",command,
            "--manifest",w["root"]+"/manifest.json","--worker-id",str(w["id"]),
            "--output",w["root"]+"/results"]+(extra or [])

for exe in ["ssh","scp","git"]:
    if not shutil.which(exe):raise RuntimeError(f"Install {exe} on this controller first")
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT), "Use an exact commit SHA"
COMMIT=COMMIT.lower()
assert re.fullmatch(r"[A-Za-z0-9_-]+",CAMPAIGN)
assert 0 < CORE_BUDGET_HOURS <= 18
assert sorted(w["id"] for w in WORKERS)==[0,1,2,3]
assert len({(w["host"],w["port"],w["gpu"]) for w in WORKERS})==4, "Duplicate endpoint/GPU"
assert re.fullmatch(r"(?:git@github\.com:[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+|https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)",REPO_URL), "Use a GitHub URL without credentials"
for w in WORKERS:
    assert re.fullmatch(r"[A-Za-z0-9_.-]+",w["host"]) and not w["host"].startswith("-") and "GPU_HOST" not in w["host"]
    assert re.fullmatch(r"[A-Za-z0-9_-]+",w["user"])
    assert isinstance(w["port"],int) and 1<=w["port"]<=65535
    assert re.fullmatch(r"[0-9]+",w["gpu"]), "Set one physical GPU index per worker"
    assert re.fullmatch(r"[A-Za-z0-9_./-]+",w["remote_base"]) and w["remote_base"].startswith("/")
    assert ".." not in Path(w["remote_base"]).parts and len(Path(w["remote_base"]).parts)>=3
    w["root"]=w["remote_base"].rstrip("/")+f"/{CAMPAIGN}_{COMMIT[:12]}/worker{w['id']}"
LOCAL_ROOT=LOCAL_ROOT.expanduser().resolve()/f"{CAMPAIGN}_{COMMIT[:12]}"
LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
MANIFEST_BYTES=(json.dumps(CORE_MANIFEST,indent=2)+"\n").encode()
MANIFEST_SHA=hashlib.sha256(MANIFEST_BYTES).hexdigest()
(LOCAL_ROOT/"manifest.json").write_bytes(MANIFEST_BYTES)
print("Core jobs:",len(CORE_MANIFEST["jobs"]),"| queues:",{i:sum(j["worker_id"]==i for j in CORE_MANIFEST["jobs"]) for i in range(4)})


## 3. Fetch the exact GitHub snapshot and verify connections

This creates an isolated local Git cache and exports the chosen commit; it never modifies or pushes your working repo. Private GitHub SSH keys stay on your laptop. Only the source archive is sent to workers.


In [ ]:
cache=LOCAL_ROOT/"git_cache"
if not cache.exists(): checked(["git","clone","--filter=blob:none","--no-checkout",REPO_URL,str(cache)],timeout=600)
actual_url=checked(["git","-C",str(cache),"remote","get-url","origin"]).strip()
assert actual_url==REPO_URL, "Git cache belongs to another repo"
checked(["git","-C",str(cache),"fetch","origin",COMMIT],timeout=600)
resolved=checked(["git","-C",str(cache),"rev-parse",COMMIT+"^{commit}"]).strip()
assert resolved==COMMIT
ARCHIVE=LOCAL_ROOT/"source.tar"
checked(["git","-C",str(cache),"archive","--format=tar",COMMIT,"-o",str(ARCHIVE)],timeout=120)
ARCHIVE_SHA=hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
with tarfile.open(ARCHIVE) as tf:
    names=set(tf.getnames())
required={"automode/rebuttal.py","requirements-rebuttal.lock","tests/test_rebuttal_invariants.py","pyproject.toml"}
assert required<=names, f"Claude implementation is incomplete; missing {required-names}"
print("Pinned commit:",COMMIT,"| source archive SHA256:",ARCHIVE_SHA)
print(each_worker(lambda w:remote(w,[w["python"],"-c","import sys; print(sys.version)"])))


## 4. Deploy source and set the gated-model token

Enter a read-only Hugging Face token if these machines are not already authenticated. It is entered invisibly, sent over SSH, and stored in a mode-600 environment file outside results; it is not part of the source archive or downloads. You must already have accepted Gemma's access terms. Press Enter to use each machine's existing Hugging Face authentication. Do not paste keys or tokens into source cells.


In [ ]:
import getpass
hf_token=getpass.getpass("HF read token (blank = use existing remote authentication): ").strip()
metadata={"commit":COMMIT,"archive_sha256":ARCHIVE_SHA,"manifest_sha256":MANIFEST_SHA}
(root_helper:=LOCAL_ROOT/"remote_job.py").write_text(REMOTE_JOB_SOURCE)

def deploy(w):
    root=w["root"]
    remote_py(w,"import json,sys,pathlib; p=pathlib.Path(json.load(sys.stdin)['root']); p.mkdir(parents=True,exist_ok=True)",{"root":root})
    # Immutable deployment: refuse to replace an existing different snapshot.
    old=remote_py(w,"import json,sys,pathlib; p=pathlib.Path(json.load(sys.stdin)['path']); print(p.read_text() if p.exists() else '{}')",{"path":root+"/source_metadata.json"})
    if json.loads(old) and json.loads(old)!=metadata: raise RuntimeError("Existing deployment differs; choose a new campaign")
    transfer(w,ARCHIVE,root+"/source.tar")
    transfer(w,root_helper,root+"/remote_job.py")
    script="""
import hashlib,json,os,pathlib,sys,tarfile,shutil
p=json.load(sys.stdin); root=pathlib.Path(p['root']); archive=root/'source.tar'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==p['metadata']['archive_sha256']
repo=root/'repo'
if not repo.exists():
    stage=root/'repo_unpack'
    if stage.exists(): raise RuntimeError('Previous unpack interrupted; inspect repo_unpack before retry')
    stage.mkdir()
    with tarfile.open(archive) as tf:
        for member in tf.getmembers():
            dest=(stage/member.name).resolve()
            if not dest.is_relative_to(stage.resolve()) or not (member.isdir() or member.isfile()):
                raise RuntimeError('Unsupported archive path/type: '+member.name)
            if member.isdir():dest.mkdir(parents=True,exist_ok=True)
            else:
                dest.parent.mkdir(parents=True,exist_ok=True)
                with tf.extractfile(member) as src, dest.open('wb') as dst:shutil.copyfileobj(src,dst)
                os.chmod(dest,member.mode & 0o777)
    stage.rename(repo)
(root/'manifest.json').write_text(p['manifest'])
(root/'source_metadata.json').write_text(json.dumps(p['metadata'],indent=2))
envfile=root/'worker_env.json'
if envfile.exists() and 'HF_TOKEN' not in p['env']:
    previous=json.loads(envfile.read_text())
    if previous.get('HF_TOKEN'):p['env']['HF_TOKEN']=previous['HF_TOKEN']
fd=os.open(envfile,os.O_WRONLY|os.O_CREAT|os.O_TRUNC,0o600)
os.fchmod(fd,0o600)
with os.fdopen(fd,'w') as f:json.dump(p['env'],f)
print('deployed')
"""
    env={"CUDA_VISIBLE_DEVICES":w["gpu"],"TOKENIZERS_PARALLELISM":"false","PYTHONUNBUFFERED":"1","MPLBACKEND":"Agg","AUTOMODE_CODE_COMMIT":COMMIT,"AUTOMODE_SOURCE_METADATA":root+"/source_metadata.json","OMP_NUM_THREADS":"4"}
    if hf_token:env["HF_TOKEN"]=hf_token
    return remote_py(w,script,{"root":root,"metadata":metadata,"manifest":MANIFEST_BYTES.decode(),"env":env})
try:print(each_worker(deploy))
finally:hf_token=""


## 5. Bootstrap the four isolated Python environments

This starts background installation on each worker. It uses the rental image's existing PyTorch/CUDA, creates an isolated per-worker venv with access to those packages, installs Claude's pinned application dependencies, and verifies that torch did not change. An incompatible rental image fails here; the notebook does not replace drivers.


In [ ]:
def bootstrap(w):
    root=w["root"]; py=root+"/venv/bin/python"
    script="\n".join([
        "set -euo pipefail", "umask 077",
        f"{q(w['python'])} -c "+q("import torch; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.__version__) ")+f" > {q(root+'/torch_before.txt')}",
        f"test -d {q(root+'/venv')} || {q(w['python'])} -m venv --system-site-packages {q(root+'/venv')}",
        f"{q(py)} -m pip install -r requirements-rebuttal.lock",
        f"{q(py)} -m pip install --no-deps -e .",
        f"{q(py)} -c "+q("import torch; assert torch.cuda.is_available(); print(torch.__version__)")+f" > {q(root+'/torch_after.txt')}",
        f"cmp {q(root+'/torch_before.txt')} {q(root+'/torch_after.txt')}",
        f"{q(py)} -m automode.rebuttal contract > {q(root+'/contract.json')}",
    ])
    return launch(w,"bootstrap",["bash","-c",script],1.5)
print(each_worker(bootstrap))


## 6. Check bootstrap, then start preflight

If installation is still running, use the monitoring cell below and rerun this cell later. The gates stop execution rather than pretending that a failed install or test succeeded.


In [ ]:
require_completed("bootstrap")
contracts=each_worker(lambda w:read_remote_json(w,w["root"]+"/contract.json"))
for wid,c in contracts.items():
    assert c.get("api_version")=="1.0", (wid,"wrong API")
    assert REQUIRED_METHODS<=set(c.get("supported_methods",[])), (wid,"missing methods")
    assert c.get("metric_key")=="eval_gsm8k_maj3"
print(each_worker(lambda w:launch(w,"preflight",cli(w,"preflight"),1.5)))


## 7. Check all preflights, then start pilots

Preflight verifies tests, GPU memory, model/data access and immutable revisions. The physical UUID check also catches two different SSH endpoints that accidentally expose the same GPU. Every worker uses the same model/data versions and dependency lock.


In [ ]:
require_completed("preflight")
preflights=each_worker(lambda w:read_remote_json(w,w["root"]+"/results/preflight_report.json"))
for wid,p in preflights.items():
    assert p.get("status")=="ok" and p.get("tests_passed") is True
    assert p.get("worker_id")==wid and p.get("commit")==COMMIT and p.get("manifest_sha256")==MANIFEST_SHA
    assert p.get("archive_sha256")==ARCHIVE_SHA
    assert p.get("vram_gib",0)>=CORE_MANIFEST["minimum_vram_gib"]
assert len({p["gpu_uuid"] for p in preflights.values()})==4, "Workers share a physical GPU"
for field in ["model_revision","dataset_revision","dependency_lock_sha256","torch_version"]:
    assert all(p.get(field) for p in preflights.values()) and len({p[field] for p in preflights.values()})==1, f"Workers disagree on {field}"
display(list(preflights.values()))
print(each_worker(lambda w:launch(w,"pilot",cli(w,"pilot"),2.0)))


## 8. Review pilots and launch the core

Keep **LAUNCH_CORE=False** until you have inspected the four reports. A pilot is a runtime/memory check, not paper evidence. The ETA check reserves 15% of the configured core window. If it fails, improve the throughput estimate or reduce the experiment plan explicitly; do not shorten epochs or test data silently.

To launch: set LAUNCH_CORE=True in the settings cell, rerun that settings cell only with the same values, rerun helpers if needed to restore derived paths, then rerun this cell. Simpler: execute `LAUNCH_CORE = True` in a new cell after reviewing the reports. The main launch runs ONLY the 15 planned experiments; it does not start optional sweeps.


In [ ]:
require_completed("pilot")
pilots=each_worker(lambda w:read_remote_json(w,w["root"]+"/results/pilot_report.json"))
for wid,p in pilots.items():
    assert p.get("status")=="ok" and p.get("worker_id")==wid and p.get("n_eval")==100
    assert p.get("commit")==COMMIT and p.get("manifest_sha256")==MANIFEST_SHA
    assert p.get("estimated_core_queue_hours",0)>0
    if wid<3:assert p.get("post_promotion_optimizer_steps",0)>=5
    assert p["estimated_core_queue_hours"]<=CORE_BUDGET_HOURS*.85, f"Worker {wid} ETA exceeds the available window"
display(list(pilots.values()))
if LAUNCH_CORE:
    # Re-check preflight success rather than rely on notebook variables from a prior session.
    require_completed("preflight")
    print(each_worker(lambda w:launch(w,"core",cli(w,"run",["--hours",str(CORE_BUDGET_HOURS)]),CORE_BUDGET_HOURS+1)))
else:
    print("Pilot review only. Set LAUNCH_CORE = True in a new cell, then rerun this cell to start training.")


## 9. Monitor at any time

Set PHASE to bootstrap/preflight/pilot/core. Run this cell whenever you want an update; there is no notebook keepalive loop. On reconnect, rerun settings/helpers with identical values, then use this cell. If a job failed, read its log and correct the cause. New code requires a new pinned commit/deployment; do not edit running source. An explicit `launch(..., retry=True)` retries a failed phase using the same command, but only verified completed experiments may be skipped by the training CLI.


In [ ]:
PHASE="core"
assert PHASE in {"bootstrap","preflight","pilot","core"}
def monitor(w):
    state=job_status(w,PHASE)
    tail=remote_py(w,"import json,sys,pathlib; p=pathlib.Path(json.load(sys.stdin)['path']); print(''.join(p.read_text(errors='replace').splitlines(keepends=True)[-18:]) if p.exists() else '(no log yet)')",{"path":w["root"]+"/jobs/"+PHASE+"/output.log"})
    return state,tail
for wid,(state,tail) in sorted(each_worker(monitor).items()):
    print(f"\nWORKER {wid}: {state}")
    print(tail)


## 10. Download logs, predictions and reports

Run during or after training. Partial downloads are labeled by their timestamp; rerun after completion for final artifacts. Checkpoints and credentials are excluded. Full checkpoints remain on the rental disk and must be copied separately before terminating the machines. Downloads do not stop billing.


In [ ]:
EXPORT_ROOT=LOCAL_ROOT/"downloads"/time.strftime("%Y%m%d_%H%M%S",time.gmtime())
EXPORT_ROOT.mkdir(parents=True,exist_ok=False)
def download(w):
    archive=w["root"]+"/results_export.tar.gz"
    script="""
import json,pathlib,sys,tarfile
p=json.load(sys.stdin);root=pathlib.Path(p['root'])
with tarfile.open(p['archive'],'w:gz') as tf:
    for name in ['results','jobs','source_metadata.json','manifest.json','contract.json']:
        base=root/name
        if not base.exists():continue
        entries=[base] if base.is_file() else sorted(base.rglob('*'))
        for f in entries:
            if f.is_symlink() or not f.is_file() or f.name.endswith('.tmp'):continue
            rel=f.relative_to(root)
            if 'checkpoints' in rel.parts or f.suffix.lower() in {'.safetensors','.bin','.pt','.pth'}:continue
            tf.add(f,arcname=str(rel),recursive=False)
print('exported')
"""
    remote_py(w,script,{"root":w["root"],"archive":archive},timeout=600)
    local=EXPORT_ROOT/f"worker{w['id']}.tar.gz"
    transfer(w,archive,local,download=True)
    dest=EXPORT_ROOT/f"worker{w['id']}";dest.mkdir()
    with tarfile.open(local) as tf:
        for member in tf.getmembers():
            target=(dest/member.name).resolve()
            if not target.is_relative_to(dest.resolve()) or not member.isfile():
                raise RuntimeError("Unexpected downloaded archive entry")
            target.parent.mkdir(parents=True,exist_ok=True)
            with tf.extractfile(member) as src,target.open('wb') as dst:shutil.copyfileobj(src,dst)
    return str(dest)
print(each_worker(download))
print("Saved locally:",EXPORT_ROOT)


## 11. Validate and summarize completed runs

This reads the latest download, verifies commit/manifest/counts, and reports mean and sample SD across seeds for each exact method. Incomplete rows remain visible. It does not calculate significance from three seeds or merge historical runs. Percentages below come only from actual completed ledger rows.


In [ ]:
rows=[]; expected={j["job_id"]:j for j in CORE_MANIFEST["jobs"]}
for w in WORKERS:
    p=EXPORT_ROOT/f"worker{w['id']}"/"results/run_ledger.json"
    if not p.exists():
        print(f"Worker {w['id']}: no run ledger yet");continue
    ledger=json.loads(p.read_text())
    assert ledger["worker_id"]==w["id"] and ledger["commit"]==COMMIT and ledger["manifest_sha256"]==MANIFEST_SHA
    for r in ledger.get("runs",[]):
        assert r["job_id"] in expected
        job=expected[r["job_id"]]
        assert job["worker_id"]==w["id"] and r["seed"]==job["overrides"]["seed"] and r["method"]==job["overrides"]["method"]
        if r["status"]=="completed":
            assert r["eval_gsm8k_n_total"]==1319 and 0<=r["eval_gsm8k_maj3"]<=1
        rows.append(r)
assert len({r['job_id'] for r in rows})==len(rows), "Duplicate job IDs"
completed=[r for r in rows if r["status"]=="completed"]
print(f"Verified completed: {len(completed)}/15")
for method in sorted(REQUIRED_METHODS):
    subset=[r for r in completed if r["method"]==method]
    scores=[100*r["eval_gsm8k_maj3"] for r in subset]
    print(method, "n=",len(scores),"mean=",round(statistics.mean(scores),3) if scores else None,
          "sample_SD=",round(statistics.stdev(scores),3) if len(scores)>1 else None,
          "seeds=",sorted(r["seed"] for r in subset))
print("Missing/incomplete:",[j for j in expected if j not in {r['job_id'] for r in completed}])
if rows:
    fields=sorted({k for r in rows for k in r})
    summary=EXPORT_ROOT/"verified_results.csv"
    with summary.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)
    display(FileLink(str(summary)))


## 12. Stop only a selected job, if needed

This sends a job-specific stop request to the active supervisor, which terminates its own training subprocess group. It does not kill unrelated GPU processes and does not release the rental. Nothing happens with WORKER_TO_STOP=None. A stopped experiment is incomplete, not a result. Download logs and full checkpoints before terminating the rental through its provider.


In [ ]:
WORKER_TO_STOP=None  # Explicitly set 0, 1, 2, or 3 to stop one worker.
PHASE_TO_STOP="core"
if WORKER_TO_STOP is not None:
    assert WORKER_TO_STOP in {0,1,2,3} and PHASE_TO_STOP in {"bootstrap","preflight","pilot","core"}
    w=next(w for w in WORKERS if w["id"]==WORKER_TO_STOP)
    print(remote(w,[w["python"],w["root"]+"/remote_job.py","stop","--job-dir",w["root"]+"/jobs/"+PHASE_TO_STOP]))
else:
    print("No stop requested.")


## Finish

Keep the source commit, manifest, lockfile, successful reports, full per-item predictions, transition logs and verified full checkpoints together. Changes to code or scientific settings require a new campaign identity and new results. Discuss actual findings, including negative controls. No submission, email, GitHub push or rental action is performed by this notebook.

The launcher and process supervisor can be tested without GPUs; model correctness, CUDA compatibility, checkpoint equivalence and throughput still require Claude's implementation and the rental preflight/pilot. See the kit's VALIDATION.md for what was actually tested before delivery.
